In [1]:
!pip install torch torchvision

import os
import torch
torch.cuda.empty_cache()

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets,transforms,models
from torchvision.models import DenseNet121_Weights
from torch.utils.data import DataLoader
import os

In [4]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
data_transforms ={
    "Training":transforms.Compose([
        transforms.RandomResizedCrop(224,scale=(0.85, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomAffine(degrees=10, translate=(0.05, 0.05)),
        transforms.RandomApply([
            transforms.GaussianBlur(kernel_size=(5, 9), sigma=(0.1, 2.0))
        ], p=0.1),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ]),
    "Testing":
    transforms.Compose([
        transforms.Resize(224),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ]),
}

In [6]:
dir = "Dataset/BT-MRI Dataset/BT-MRI Dataset"

In [7]:
image_data = {
    x: datasets.ImageFolder(os.path.join(dir, x), data_transforms[x])
    for x in ["Training", "Testing"]
}

In [8]:
print("Successfully loaded datasets!")
print(f"Training images: {len(image_data['Training'])}")
print(f"Testing images: {len(image_data['Testing'])}")

Successfully loaded datasets!
Training images: 5712
Testing images: 1311


In [9]:
dataloaders = {
    x: DataLoader(image_data[x], batch_size=16, shuffle=True, num_workers=4)
    for x in ["Training", "Testing"]
}

In [10]:
model = models.densenet121(weights=DenseNet121_Weights.DEFAULT)

In [10]:
for params in model.parameters():
    params.requires_grad=False

In [11]:
num_classes = 4
in_features = model.classifier.in_features
model.classifier = nn.Linear(in_features, num_classes)
model = model.to(device)

In [12]:
loss_ = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(),lr=0.001)
epochs = 5

In [17]:
for param in model.classifier.parameters():
    param.requires_grad = True

# Check what device PyTorch thinks it's using
print(f"Current Device: {device}")

# Check where the model actually lives
print(f"Model is on: {next(model.parameters()).device}")

for i in range(epochs+1):
    model.train()
    running_loss=0.0
    for image,label in dataloaders["Training"]:
        image,label = image.to(device),label.to(device)
        optimizer.zero_grad()

        output = model(image)
        loss = loss_(output,label)
        loss.backward()
        optimizer.step()

        running_loss +=loss.item()*image.size(0)
    epoch_loss = running_loss/len(image_data["Training"])
    print(f"Epoch{i+1}/{epochs} - Loss {epoch_loss:.4f}")

Current Device: cuda
Model is on: cuda:0
Epoch1/5 - Loss 0.3782


KeyboardInterrupt: 

In [18]:
for i in model.parameters():
    i.requires_grad=True

In [19]:
optimizer_ft = optim.Adam(model.classifier.parameters(),lr=1e-4)
epochs_ft = 30
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer_ft,T_max=epochs_ft,eta_min=1e-6)

In [20]:
print(f"Current Device: {device}")
print(f"Model is on: {next(model.parameters()).device}")
for epoch in range(epochs_ft):
    model.train()
    running_loss = 0.0
    
    for image, label in dataloaders["Training"]:
        image, label = image.to(device), label.to(device)
        
        optimizer_ft.zero_grad()
        outputs = model(image)
        loss = loss_(outputs, label)
        loss.backward()
        optimizer_ft.step()
        
        running_loss += loss.item() * image.size(0)
        
    epoch_loss = running_loss / len(image_data["Training"])
    print(f'Fine-Tune Epoch {epoch+1}/{epochs_ft} - Loss: {epoch_loss:.4f}')

Current Device: cuda
Model is on: cuda:0
Fine-Tune Epoch 1/30 - Loss: 0.3345
Fine-Tune Epoch 2/30 - Loss: 0.3252
Fine-Tune Epoch 3/30 - Loss: 0.3223
Fine-Tune Epoch 4/30 - Loss: 0.3115
Fine-Tune Epoch 5/30 - Loss: 0.3360
Fine-Tune Epoch 6/30 - Loss: 0.3328
Fine-Tune Epoch 7/30 - Loss: 0.3277
Fine-Tune Epoch 8/30 - Loss: 0.3198
Fine-Tune Epoch 9/30 - Loss: 0.3219
Fine-Tune Epoch 10/30 - Loss: 0.3278
Fine-Tune Epoch 11/30 - Loss: 0.3274
Fine-Tune Epoch 12/30 - Loss: 0.3220
Fine-Tune Epoch 13/30 - Loss: 0.3207
Fine-Tune Epoch 14/30 - Loss: 0.3140
Fine-Tune Epoch 15/30 - Loss: 0.3167
Fine-Tune Epoch 16/30 - Loss: 0.3244
Fine-Tune Epoch 17/30 - Loss: 0.3273
Fine-Tune Epoch 18/30 - Loss: 0.3237
Fine-Tune Epoch 19/30 - Loss: 0.3289
Fine-Tune Epoch 20/30 - Loss: 0.3234
Fine-Tune Epoch 21/30 - Loss: 0.3139
Fine-Tune Epoch 22/30 - Loss: 0.3218
Fine-Tune Epoch 23/30 - Loss: 0.3101
Fine-Tune Epoch 24/30 - Loss: 0.3090
Fine-Tune Epoch 25/30 - Loss: 0.3184
Fine-Tune Epoch 26/30 - Loss: 0.3184
Fine-T

In [21]:
model.eval()
correct = 0
total = 0
print("Evaluating on the standard Testing dataset...")
with torch.no_grad():
    for images, labels in dataloaders["Testing"]:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Accuracy on clean Testing data: {accuracy:.2f}%')

Evaluating on the standard Testing dataset...
Accuracy on clean Testing data: 87.57%


In [22]:
import os
from torch.utils.data import DataLoader

challenging_base_dir = "Dataset/Challenging Datasets/Challenging Datasets"

challenging_folders = [
    "Blurred Dataset", 
    "Noisy Dataset", 
    "Patient Motion Artifact Dataset"
]

print("--- Starting Stress Test on Challenging Data ---")

for folder_name in challenging_folders:
    folder_path = os.path.join(challenging_base_dir, folder_name)
    test_dataset = datasets.ImageFolder(folder_path, data_transforms['Testing'])
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
    
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    accuracy = 100 * correct / total
    print(f"Accuracy on {folder_name}: {accuracy:.2f}%")

--- Starting Stress Test on Challenging Data ---
Accuracy on Blurred Dataset: 83.60%
Accuracy on Noisy Dataset: 42.75%
Accuracy on Patient Motion Artifact Dataset: 31.84%


In [23]:
import torch

print("Saving model for Streamlit Web App...")

# 1. Define your save path
streamlit_model_path = "densenet121_brain_tumor.pth"

# 2. Save only the learned weights (Best Practice)
torch.save(model.state_dict(), streamlit_model_path)

print(f"Success! Streamlit model saved as: {streamlit_model_path}")

Saving model for Streamlit Web App...
Success! Streamlit model saved as: densenet121_brain_tumor.pth


In [ ]:
import torch

# 1. Load your champion weights
model.load_state_dict(torch.load("densenet121_brain_tumor.pth", map_location='cpu'))
model.to('cpu')


# 2. Convert to Half-Precision (Float16)
# Note: On some older CPUs, we stay in Float32 but save as Float16 to save space
model_half = model.half() 

# 3. Save the weights
torch.save(model_half.state_dict(), "densenet121_fp16.pth")

print(f"FP16 Model Saved! Size: {os.path.getsize('densenet121_fp16.pth')/1e6:.2f} MB")

FP16 Model Saved! Size: 14.35 MB


/tmp/ipykernel_55348/337588288.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("densenet121_brain_tumor.pth", map_location='cpu'))


In [35]:
import torch
import time
import os
from torchvision import models
from torch.utils.data import DataLoader
import torch.nn as nn

# 1. Load the Original (32-bit) Model
model_fp32 = models.densenet121(weights=None)
model_fp32.classifier = nn.Linear(model_fp32.classifier.in_features, 4)
model_fp32.load_state_dict(torch.load("densenet121_brain_tumor.pth", map_location='cpu'))
model_fp32.eval()

# 2. Load the New (16-bit) Model
# Even though the file is FP16, we load it into an FP32 structure for CPU stability
model_fp16 = models.densenet121(weights=None)
model_fp16.classifier = nn.Linear(model_fp16.classifier.in_features, 4)
state_dict_fp16 = torch.load("densenet121_fp16.pth", map_location='cpu')

# Convert the 16-bit weights back to 32-bit in memory so the CPU can read them
for key in state_dict_fp16:
    state_dict_fp16[key] = state_dict_fp16[key].float()

model_fp16.load_state_dict(state_dict_fp16)
model_fp16.eval()

# 3. Benchmark Function
def test_performance(model_to_test):
    correct = 0
    total = 0
    start = time.perf_counter()
    with torch.no_grad():
        for imgs, labels in dataloaders["Testing"]:
            imgs, labels = imgs.to('cpu'), labels.to('cpu')
            outputs = model_to_test(imgs)
            _, pred = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (pred == labels).sum().item()
    
    latency = ((time.perf_counter() - start) / total) * 1000
    accuracy = 100 * correct / total
    return accuracy, latency

# 4. Run Tests
print("Testing Original Model (27MB)...")
acc32, lat32 = test_performance(model_fp32)

print("Testing FP16 Model (14MB)...")
acc16, lat16 = test_performance(model_fp16)

# 5. The Verdict
print("\n" + "="*40)
print(f"{'Metric':<15} | {'Original':<10} | {'FP16 Saved'}")
print("-" * 40)
print(f"{'Accuracy':<15} | {acc32:>8.2f}% | {acc16:>8.2f}%")
print(f"{'Latency/Img':<15} | {lat32:>8.2f}ms | {lat16:>8.2f}ms")
print(f"{'File Size':<15} | {'27.13MB':<10} | {'14.35MB'}")
print("="*40)

if abs(acc32 - acc16) < 0.01:
    print("✅ SUCCESS: The models are mathematically identical!")

Testing Original Model (27MB)...


/tmp/ipykernel_55348/1925011399.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_fp32.load_state_dict(torch.load("densenet121_brain_tumor.pth", map_location='cpu')

Testing FP16 Model (14MB)...

Metric          | Original   | FP16 Saved
----------------------------------------
Accuracy        |    87.57% |    87.49%
Latency/Img     |    49.80ms |    50.33ms
File Size       | 27.13MB    | 14.35MB


In [11]:
import torch
from torchvision import models
import torch.nn as nn
import os

# 1. Initialize a fresh 32-bit model
model_mobile = models.densenet121(weights=None)
model_mobile.classifier = nn.Linear(model_mobile.classifier.in_features, 4)

# 2. Load the ORIGINAL 27MB weights (Standard Float32)
# Keep the Lite export in Float32. The earlier FP16 mobile artifact hit
# an XNNPACK prepacked convolution error for DenseNet's stem convolution.
state_dict = torch.load("densenet121_brain_tumor.pth", map_location='cpu', weights_only=True)
model_mobile.load_state_dict(state_dict)
model_mobile.eval()

# 3. Script and freeze the model for a Lite-safe export
print("Exporting DenseNet121 for the Lite interpreter...")
scripted_model = torch.jit.script(model_mobile)
frozen_model = torch.jit.freeze(scripted_model)

# 4. Save for Lite Interpreter
mobile_final_name = "brain_tumor_final_universal.ptl"
frozen_model._save_for_lite_interpreter(mobile_final_name)

print(f"✅ Model saved: {mobile_final_name}")

# --- 5. THE ULTIMATE TEST ---
print("Running final verification on PC CPU...")
final_test = torch.jit.load(mobile_final_name)
img, _ = next(iter(dataloaders['Testing']))

# Ensure input is standard Float32
with torch.no_grad():
    output = final_test(img[0:1].float())

print(f"Success! Prediction output shape: {output.shape}")
print(f"Final Mobile File Size: {os.path.getsize(mobile_final_name)/1e6:.2f} MB")

Exporting DenseNet121 for the Lite interpreter...
✅ Model saved: brain_tumor_final_universal.ptl
Running final verification on PC CPU...
Success! Prediction output shape: torch.Size([1, 4])
Final Mobile File Size: 28.44 MB


In [12]:
# Smoke-test the final Lite artifact from a fresh load
mobile_test = torch.jit.load("brain_tumor_final_universal.ptl")
img, _ = next(iter(dataloaders['Testing']))

with torch.no_grad():
    output = mobile_test(img[0:1].float())

print(f"Mobile Output Shape: {output.shape} (Should be 1, 4)")

Mobile Output Shape: torch.Size([1, 4]) (Should be 1, 4)


In [14]:
import time
import os
import torch
import torch.nn as nn
from torchvision import models
from torch.utils.data import DataLoader

# Compare the exported Lite model against the original checkpoint model
actual_model = models.densenet121(weights=None)
actual_model.classifier = nn.Linear(actual_model.classifier.in_features, 4)
actual_state_dict = torch.load("densenet121_brain_tumor.pth", map_location='cpu', weights_only=True)
actual_model.load_state_dict(actual_state_dict)
actual_model.eval()

mobile_model = torch.jit.load("brain_tumor_final_universal.ptl")
mobile_model.eval()

test_dataset = dataloaders['Testing'].dataset
benchmark_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=0)

warmup_imgs, _ = next(iter(benchmark_loader))
warmup_imgs = warmup_imgs[0:1].float()

with torch.no_grad():
    _ = actual_model(warmup_imgs)
    _ = mobile_model(warmup_imgs)

def benchmark_model(model_to_test, loader):
    correct = 0
    total = 0
    start = time.perf_counter()
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.float().cpu()
            labels = labels.cpu()
            outputs = model_to_test(imgs)
            _, preds = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

    elapsed = time.perf_counter() - start
    accuracy = 100 * correct / total
    latency = (elapsed / total) * 1000
    return accuracy, latency

def compare_predictions(eager_model, lite_model, loader, num_batches=5):
    same_predictions = 0
    total = 0
    max_logit_diff = 0.0

    with torch.no_grad():
        for batch_idx, (imgs, _) in enumerate(loader):
            if batch_idx >= num_batches:
                break

            imgs = imgs.float().cpu()
            eager_outputs = eager_model(imgs)
            lite_outputs = lite_model(imgs)

            same_predictions += (eager_outputs.argmax(dim=1) == lite_outputs.argmax(dim=1)).sum().item()
            total += imgs.size(0)
            max_logit_diff = max(max_logit_diff, (eager_outputs - lite_outputs).abs().max().item())

    agreement = 100 * same_predictions / total if total else 0.0
    return agreement, max_logit_diff

print("Benchmarking original checkpoint model...")
actual_acc, actual_latency = benchmark_model(actual_model, benchmark_loader)

print("Benchmarking exported Lite model...")
mobile_acc, mobile_latency = benchmark_model(mobile_model, benchmark_loader)

agreement, max_logit_diff = compare_predictions(actual_model, mobile_model, benchmark_loader)

actual_size_mb = os.path.getsize("densenet121_brain_tumor.pth") / 1e6
mobile_size_mb = os.path.getsize("brain_tumor_final_universal.ptl") / 1e6

print("\n" + "=" * 64)
print(f"{'Metric':<22} | {'Actual Model':<16} | {'Mobile Lite Model'}")
print("-" * 64)
print(f"{'Accuracy':<22} | {actual_acc:>14.2f}% | {mobile_acc:>16.2f}%")
print(f"{'Latency / Image':<22} | {actual_latency:>12.2f} ms | {mobile_latency:>14.2f} ms")
print(f"{'File Size':<22} | {actual_size_mb:>12.2f} MB | {mobile_size_mb:>14.2f} MB")
print(f"{'Prediction Agreement':<22} | {'-':>14} | {agreement:>16.2f}%")
print(f"{'Max Logit Difference':<22} | {'-':>14} | {max_logit_diff:>16.6f}")
print("=" * 64)


Benchmarking original checkpoint model...
Benchmarking exported Lite model...

Metric                 | Actual Model     | Mobile Lite Model
----------------------------------------------------------------
Accuracy               |          87.57% |            87.57%
Latency / Image        |       110.83 ms |          77.73 ms
File Size              |        28.45 MB |          28.44 MB
Prediction Agreement   |              - |           100.00%
Max Logit Difference   |              - |         0.000039
